# Task 5: Quantization & Evaluation (Workbench)

Merge LoRA → Quantize to INT4 (GPTQ via llm-compressor) → Quality/Speed comparison.
Skips steps if outputs already exist.

## Setup

In [ ]:
import os, sys, json, re, time, torch, gc
import numpy as np
from PIL import Image
from collections import defaultdict, Counter
import matplotlib.pyplot as plt

sys.path.insert(0, '../../task4-fine-tuning/granulometry')
from config import *

LORA_PATH = '../../task4-fine-tuning/granulometry/lora_augmented'
MERGED_PATH = './merged_model'
QUANTIZED_PATH = './quantized_model_int4'

for i in range(torch.cuda.device_count()):
    print(f'GPU {i}: {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB')


## Step 1: Merge LoRA into Base Model

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

if os.path.exists(MERGED_PATH) and os.path.isdir(MERGED_PATH):
    merged_size = sum(f.stat().st_size for f in os.scandir(MERGED_PATH) if f.is_file()) / 1e9
    print(f'Merged model already exists: {merged_size:.2f} GB (delete folder to re-merge)')
else:
    from peft import PeftModel
    print('Loading base model...')
    base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID, device_map='auto', torch_dtype=torch.bfloat16)
    print(f'Loading LoRA adapter from {LORA_PATH}...')
    model = PeftModel.from_pretrained(base_model, LORA_PATH)
    print('Merging...')
    model = model.merge_and_unload()
    print(f'Saving to {MERGED_PATH}...')
    model.save_pretrained(MERGED_PATH)
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    processor.save_pretrained(MERGED_PATH)
    merged_size = sum(f.stat().st_size for f in os.scandir(MERGED_PATH) if f.is_file()) / 1e9
    print(f'Merged model: {merged_size:.2f} GB')
    del model, base_model; gc.collect(); torch.cuda.empty_cache()


## Step 2: Quantize to INT4 (GPTQ via llm-compressor)

In [ ]:
if os.path.exists(QUANTIZED_PATH) and os.path.isdir(QUANTIZED_PATH):
    quant_size = sum(f.stat().st_size for f in os.scandir(QUANTIZED_PATH) if f.is_file()) / 1e9
    print(f'Quantized model already exists: {quant_size:.2f} GB (delete folder to re-quantize)')
else:
    from llmcompressor.modifiers.quantization import GPTQModifier
    from llmcompressor import oneshot
    
    print('Loading merged model...')
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MERGED_PATH, device_map='auto', dtype=torch.bfloat16)
    processor = AutoProcessor.from_pretrained(MERGED_PATH)
    
    # Ignore vision encoder (not divisible by group_size 128)
    ignore_list = ['lm_head']
    for i in range(32):
        for layer in ['mlp.down_proj', 'mlp.up_proj', 'mlp.gate_proj', 'attn.qkv', 'attn.proj']:
            ignore_list.append(f'model.visual.blocks.{i}.{layer}')
    
    recipe = GPTQModifier(targets='Linear', scheme='W4A16', ignore=ignore_list)
    
    print(f'Quantizing with GPTQ W4A16 (ignoring {len(ignore_list)} vision layers)...')
    oneshot(
        model=model, recipe=recipe,
        dataset='wikitext', dataset_config_name='wikitext-2-raw-v1',
        num_calibration_samples=32, max_seq_length=256, streaming=False,
    )
    
    print('Saving...')
    model.save_pretrained(QUANTIZED_PATH)
    processor.save_pretrained(QUANTIZED_PATH)
    quant_size = sum(f.stat().st_size for f in os.scandir(QUANTIZED_PATH) if f.is_file()) / 1e9
    print(f'Quantized model: {quant_size:.2f} GB (from {merged_size:.2f} GB, {merged_size/quant_size:.1f}x compression)')
    del model; gc.collect(); torch.cuda.empty_cache()


## Step 3: Evaluate — BF16 vs INT4

In [ ]:
def parse_response(raw):
    if not raw: return None
    raw = raw.replace('<','').replace('>','')
    raw = re.sub(r'```json\s*','',raw); raw = re.sub(r'```\s*','',raw).strip()
    try:
        obj = json.loads(raw)
        if isinstance(obj, dict): return obj
    except: pass
    m = re.search(r'\{.*\}', raw, re.DOTALL)
    if m:
        try: return json.loads(m.group())
        except: pass
    sm = re.search(r'"max_particle_size_mm"\s*:\s*(\d+)', raw)
    gm = re.search(r'"grading"\s*:\s*"(\w+)"', raw)
    if sm and gm: return {'max_particle_size_mm': int(sm.group(1)), 'grading': gm.group(1)}
    return None

def run_eval(model, processor, manifest, label=''):
    model.eval()
    results=[]; cs=0; cg=0; vj=0; tt=0; peak_mem=0
    for i, entry in enumerate(manifest):
        img_path = os.path.join(TEST_DIR, entry['image'])
        if not os.path.exists(img_path): continue
        image = Image.open(img_path).convert('RGB')
        scale = min(MAX_DIM/max(image.size), 1.0)
        gsd = ORIGINAL_GSD * scale
        prompt = make_prompt(gsd)
        msgs = [{'role':'user','content':[{'type':'image','image':image},{'type':'text','text':prompt}]}]
        text = processor.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inputs = processor(text=[text], images=[image], return_tensors='pt', padding=True).to(model.device)
        torch.cuda.reset_peak_memory_stats()
        t=time.time()
        with torch.no_grad(): ids = model.generate(**inputs, max_new_tokens=128, temperature=0.1, do_sample=True)
        elapsed=time.time()-t; tt+=elapsed
        peak_mem = max(peak_mem, torch.cuda.max_memory_allocated()/1e9)
        out = processor.batch_decode(ids[:,inputs.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()
        del inputs, ids; image.close(); torch.cuda.empty_cache()
        parsed = parse_response(out)
        gs=entry['max_particle_size_mm']; gg=entry['grading']; so=False; go=False
        if parsed:
            vj+=1; ps=parsed.get('max_particle_size_mm')
            if isinstance(ps,str): ps=int(ps) if ps.isdigit() else None
            if ps==gs: so=True; cs+=1
            if parsed.get('grading','').lower()==gg: go=True; cg+=1
        results.append({'image':entry['image'],'class':entry['class'],'gt_size':gs,'gt_grading':gg,
            'predicted':parsed,'raw':out,'size_correct':so,'grading_correct':go,
            'valid_json':parsed is not None,'time_s':round(elapsed,2)})
        if (i+1)%20==0:
            n=i+1; print(f'  [{n}/{len(manifest)}] Size:{cs}/{n}({cs/n*100:.0f}%) Grade:{cg}/{n}({cg/n*100:.0f}%)')
    return results, cs, cg, vj, tt, peak_mem

with open(TEST_MANIFEST) as f: test_manifest = json.load(f)
quick = [next(e for e in test_manifest if e['class']==cls) for cls in sorted(GT.keys())]
print(f'Test: {len(test_manifest)} images, Quick: {len(quick)} images')


### BF16 Merged Model

In [ ]:
print('Loading BF16 merged model...')
proc_bf16 = AutoProcessor.from_pretrained(MERGED_PATH, min_pixels=256*28*28, max_pixels=512*28*28)
model_bf16 = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MERGED_PATH, device_map='auto', torch_dtype=torch.bfloat16)
bf16_mem = torch.cuda.memory_allocated(0)/1e9
print(f'Model VRAM: {bf16_mem:.1f} GB')

print('\nQuick eval (9):')
r9b, cs9b, cg9b, vj9b, tt9b, mem9b = run_eval(model_bf16, proc_bf16, quick)
print(f'Size: {cs9b}/{len(r9b)} | Grading: {cg9b}/{len(r9b)}')
for x in r9b:
    p=x['predicted']; ps=p.get('max_particle_size_mm','?') if p else '?'; pg=p.get('grading','?') if p else '?'
    sv='✓' if x['size_correct'] else '✗'; gv='✓' if x['grading_correct'] else '✗'
    print(f'  {x["class"]:>3} GT:{x["gt_size"]}mm/{x["gt_grading"]} Pred:{ps}mm/{pg} {sv}{gv}')

print('\nFull eval (108):')
r_bf16, cs_bf16, cg_bf16, vj_bf16, tt_bf16, mem_bf16 = run_eval(model_bf16, proc_bf16, test_manifest)
n=len(r_bf16); both_bf16=sum(1 for x in r_bf16 if x['size_correct'] and x['grading_correct'])
print(f'BF16: Size={cs_bf16}/{n}({cs_bf16/n*100:.1f}%) Grade={cg_bf16}/{n}({cg_bf16/n*100:.1f}%) Both={both_bf16}/{n}({both_bf16/n*100:.1f}%)')
print(f'Time: {tt_bf16/n:.2f}s/img | Peak VRAM: {mem_bf16:.1f} GB')
del model_bf16; gc.collect(); torch.cuda.empty_cache()


### INT4 Quantized Model (GPTQ)

In [ ]:
print('Loading INT4 quantized model...')
proc_int4 = AutoProcessor.from_pretrained(QUANTIZED_PATH, min_pixels=256*28*28, max_pixels=512*28*28)
model_int4 = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    QUANTIZED_PATH, device_map='auto')
int4_mem = torch.cuda.memory_allocated(0)/1e9
print(f'Model VRAM: {int4_mem:.1f} GB')

print('\nQuick eval (9):')
r9i, cs9i, cg9i, vj9i, tt9i, mem9i = run_eval(model_int4, proc_int4, quick)
print(f'Size: {cs9i}/{len(r9i)} | Grading: {cg9i}/{len(r9i)}')
for x in r9i:
    p=x['predicted']; ps=p.get('max_particle_size_mm','?') if p else '?'; pg=p.get('grading','?') if p else '?'
    sv='✓' if x['size_correct'] else '✗'; gv='✓' if x['grading_correct'] else '✗'
    print(f'  {x["class"]:>3} GT:{x["gt_size"]}mm/{x["gt_grading"]} Pred:{ps}mm/{pg} {sv}{gv}')

print('\nFull eval (108):')
r_int4, cs_int4, cg_int4, vj_int4, tt_int4, mem_int4 = run_eval(model_int4, proc_int4, test_manifest)
n=len(r_int4); both_int4=sum(1 for x in r_int4 if x['size_correct'] and x['grading_correct'])
print(f'INT4: Size={cs_int4}/{n}({cs_int4/n*100:.1f}%) Grade={cg_int4}/{n}({cg_int4/n*100:.1f}%) Both={both_int4}/{n}({both_int4/n*100:.1f}%)')
print(f'Time: {tt_int4/n:.2f}s/img | Peak VRAM: {mem_int4:.1f} GB')
del model_int4; gc.collect(); torch.cuda.empty_cache()


## Step 4: Comparison

In [ ]:
n = len(r_bf16)
both_bf16 = sum(1 for x in r_bf16 if x['size_correct'] and x['grading_correct'])
both_int4 = sum(1 for x in r_int4 if x['size_correct'] and x['grading_correct'])

print(f'{"Metric":<20} {"BF16":>10} {"INT4":>10} {"Delta":>12}')
print('=' * 54)
rows = [
    ('Model size (disk)', f'{merged_size:.2f} GB', f'{quant_size:.2f} GB', f'{merged_size/quant_size:.1f}x smaller'),
    ('Model VRAM', f'{bf16_mem:.1f} GB', f'{int4_mem:.1f} GB', f'{bf16_mem-int4_mem:.1f} GB saved'),
    ('Peak VRAM', f'{mem_bf16:.1f} GB', f'{mem_int4:.1f} GB', f'{mem_bf16-mem_int4:.1f} GB saved'),
    ('Avg time/image', f'{tt_bf16/n:.2f}s', f'{tt_int4/n:.2f}s', f'{(tt_bf16/n)/(tt_int4/n):.1f}x' if tt_int4>0 else '-'),
    ('FPS', f'{n/tt_bf16:.2f}', f'{n/tt_int4:.2f}', f'{(n/tt_int4)/(n/tt_bf16):.1f}x'),
    ('Size accuracy', f'{cs_bf16/n*100:.1f}%', f'{cs_int4/n*100:.1f}%', f'{(cs_int4-cs_bf16)/n*100:+.1f}%'),
    ('Grading accuracy', f'{cg_bf16/n*100:.1f}%', f'{cg_int4/n*100:.1f}%', f'{(cg_int4-cg_bf16)/n*100:+.1f}%'),
    ('Both correct', f'{both_bf16/n*100:.1f}%', f'{both_int4/n*100:.1f}%', f'{(both_int4-both_bf16)/n*100:+.1f}%'),
    ('JSON validity', f'{vj_bf16/n*100:.0f}%', f'{vj_int4/n*100:.0f}%', '-'),
]
for label, bf, it, delta in rows:
    print(f'{label:<20} {bf:>10} {it:>10} {delta:>12}')

size_drop = (cs_bf16 - cs_int4) / n * 100
grade_drop = (cg_bf16 - cg_int4) / n * 100
print(f'\nQuality degradation: size {size_drop:+.1f}%, grading {grade_drop:+.1f}%')
print(f'Target: <5% → {"PASS" if abs(size_drop)<5 and abs(grade_drop)<5 else "FAIL"}')


## Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Accuracy
metrics = ['Size', 'Grading', 'Both']
bf_vals = [cs_bf16/n*100, cg_bf16/n*100, both_bf16/n*100]
it_vals = [cs_int4/n*100, cg_int4/n*100, both_int4/n*100]
x = np.arange(len(metrics)); w = 0.35
axes[0].bar(x-w/2, bf_vals, w, label='BF16', color='steelblue')
axes[0].bar(x+w/2, it_vals, w, label='INT4', color='coral')
axes[0].set_xticks(x); axes[0].set_xticklabels(metrics)
axes[0].set_ylabel('%'); axes[0].set_title('Accuracy'); axes[0].legend(); axes[0].set_ylim(0,110)
for bars in [axes[0].containers[0], axes[0].containers[1]]:
    for bar in bars:
        axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1, f'{bar.get_height():.0f}%', ha='center', fontsize=9)

# Speed
axes[1].bar(['BF16', 'INT4'], [tt_bf16/n, tt_int4/n], color=['steelblue','coral'])
axes[1].set_ylabel('Seconds/image'); axes[1].set_title('Inference Speed')

# Memory
axes[2].bar(['BF16 disk', 'INT4 disk', 'BF16 VRAM', 'INT4 VRAM'],
    [merged_size, quant_size, mem_bf16, mem_int4], color=['steelblue','coral','steelblue','coral'])
axes[2].set_ylabel('GB'); axes[2].set_title('Size & Memory')

plt.suptitle('BF16 vs INT4 (GPTQ)', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()


## Save Results

In [ ]:
for label, r, cs, cg, vj, tt, mem in [
    ('bf16', r_bf16, cs_bf16, cg_bf16, vj_bf16, tt_bf16, mem_bf16),
    ('int4', r_int4, cs_int4, cg_int4, vj_int4, tt_int4, mem_int4),
]:
    n=len(r); both=sum(1 for x in r if x['size_correct'] and x['grading_correct'])
    with open(f'results_{label}.json','w') as f:
        json.dump({'model':MODEL_ID,'precision':label,'quantization':'GPTQ W4A16 (llm-compressor)',
            'total_images':n,'json_validity_pct':round(vj/n*100,1),
            'size_accuracy_pct':round(cs/n*100,1),'grading_accuracy_pct':round(cg/n*100,1),
            'both_correct_pct':round(both/n*100,1),'avg_inference_time_s':round(tt/n,2),
            'peak_vram_gb':round(mem,1),'results':r}, f, indent=2)
    print(f'Saved results_{label}.json')

print(f'\nQuantized model at {QUANTIZED_PATH}/ — ready for edge deployment.')
